<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0 -->

# Worker Safety in a Classical Warehouse with Cosmos 3 Reasoner

This notebook:
1. Sets up an isolated environment (`vllm` + the `vllm-cosmos3` plugin)
2. Launches a vLLM OpenAI-compatible server with the Cosmos 3 Reasoner
3. Loads the open-access [pjramg/Safe_Unsafe_Test](https://huggingface.co/datasets/pjramg/Safe_Unsafe_Test) warehouse safety dataset via FiftyOne
4. Classifies each video into one of 8 industrial safety classes using chain-of-thought reasoning
5. Evaluates accuracy and visualizes results

| Model | Params | GPUs | Context |
|-------|--------|------|---------|
| `nvidia/Cosmos3-Nano` | 16 B (8 B Reasoner) | 1× (24 GB+) | 16 384 tokens |
| `nvidia/Cosmos3-Super` | 64 B (32 B Reasoner) | 4× H100 80 GB | 262 144 tokens |

## 1. Environment Setup

### Prerequisites

1. Linux machine with NVIDIA GPU (1× GPU for Cosmos3-Nano, 4× GPUs for Cosmos3-Super)
2. Hugging Face account with access to `nvidia/Cosmos3-Super` (or `Cosmos3-Nano`)
3. Docker or a Python 3.11+ environment with `uv`

### Authenticate with Hugging Face

```bash
# Option A: interactive login
uvx hf@latest auth login

# Option B: set token in environment
export HF_TOKEN=<your_token>
```

In [ ]:
from pathlib import Path
import os
import subprocess


def find_repo_root() -> Path:
    try:
        return Path(
            subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
        ).resolve()
    except Exception:
        return Path.cwd().resolve()


REPO_ROOT = find_repo_root()
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", REPO_ROOT / "packages" / "cosmos3")).resolve()
COSMOS3_GIT_URL = os.environ.get(
    "COSMOS3_GIT_URL", "https://github.com/NVIDIA/cosmos-framework.git"
)

os.environ.setdefault("COSMOS3_REPO", str(COSMOS3_REPO))
os.environ.setdefault("COSMOS3_GIT_URL", COSMOS3_GIT_URL)
os.environ.setdefault("VLLM_PORT", "8001")
os.environ.setdefault("VLLM_LOG_FILE", "vllm_server.log")

print(f"Repo root:       {REPO_ROOT}")
print(f"Cosmos3 repo:    {COSMOS3_REPO}")
print(f"vLLM port:       {os.environ['VLLM_PORT']}")

### Install vLLM + Cosmos 3 plugin

Clone `cosmos-framework` (if not already present) and install `vllm` with the `vllm-cosmos3` + `transformers-cosmos3` plugins.

| Driver CUDA | Install |
| --- | --- |
| 13.x | `--torch-backend=cu130 "vllm==0.21.0"` (default below) |
| 12.x | `--torch-backend=cu128 "vllm==0.19.1"` |

In [ ]:
%%bash
set -euo pipefail

: "${COSMOS3_REPO:=$(git rev-parse --show-toplevel 2>/dev/null || pwd)/packages/cosmos3}"
: "${COSMOS3_GIT_URL:=https://github.com/NVIDIA/cosmos-framework.git}"

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -d "$COSMOS3_REPO/.git" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

if [ -x .venv/bin/python ]; then
  echo "Using existing venv: $PWD/.venv"
else
  echo "Creating venv..."
  uv venv --python 3.12 --seed .venv
fi

echo "Installing vLLM + Cosmos 3 plugins + recipe dependencies..."
uv pip install --python .venv/bin/python \
  --torch-backend=cu130 \
  "vllm==0.21.0" \
  "$COSMOS3_REPO/packages/transformers-cosmos3" \
  "$COSMOS3_REPO/packages/vllm-cosmos3" \
  fiftyone openai ninja

echo "Done."

## 2. Launch the vLLM Server

Start the OpenAI-compatible vLLM server in the background. The first launch compiles CUDA graphs and downloads the model checkpoint — this can take several minutes.

| Model | GPUs | `--tensor-parallel-size` | `--max-model-len` |
|-------|------|--------------------------|--------------------|
| `nvidia/Cosmos3-Super` | 4× H100 80 GB | 4 | 262144 |
| `nvidia/Cosmos3-Nano` | 1× (24 GB+) | 1 | 16384 |

In [ ]:
%%bash
export TMPDIR="/tmp/${USER:-vllm}-vllm"
export VLLM_PORT="${VLLM_PORT:-8001}"
export VLLM_LOG_FILE="${VLLM_LOG_FILE:-vllm_server.log}"
mkdir -p "$TMPDIR"

CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES:-0,1,2,3}" \
setsid .venv/bin/vllm serve nvidia/Cosmos3-Super \
  --hf-overrides '{"architectures": ["Cosmos3ReasonerForConditionalGeneration"]}' \
  --tensor-parallel-size 4 \
  --mm-encoder-tp-mode data \
  --async-scheduling \
  --allowed-local-media-path "$(pwd)" \
  --media-io-kwargs '{"video": {"num_frames": -1}}' \
  --port "$VLLM_PORT" \
  > "$VLLM_LOG_FILE" 2>&1 &

echo "vLLM server starting (PID $!, log: $VLLM_LOG_FILE)"
echo "For Cosmos3-Nano on a single GPU, change to:"
echo "  --tensor-parallel-size 1 --max-model-len 16384 --gpu-memory-utilization 0.95"

### Wait for server readiness

In [ ]:
%%bash
set -euo pipefail

PORT="${VLLM_PORT:-8001}"
LOG_FILE="${VLLM_LOG_FILE:-vllm_server.log}"

echo "Waiting for vLLM server on port ${PORT}..."
touch "$LOG_FILE"

LAST_LINE=0

for i in $(seq 1 1800); do
  TOTAL_LINES=$(wc -l < "$LOG_FILE" || echo 0)

  if [ "$TOTAL_LINES" -gt "$LAST_LINE" ]; then
    sed -n "$((LAST_LINE + 1)),$TOTAL_LINES p" "$LOG_FILE"
    LAST_LINE="$TOTAL_LINES"
  fi

  if curl -fsS "http://127.0.0.1:${PORT}/health" >/dev/null 2>&1; then
    echo "vLLM server is ready."
    exit 0
  fi

  sleep 1
done

echo "ERROR: vLLM server did not become ready within 30 minutes."
exit 1

## 3. Load the Safety Dataset

The dataset is open-access on Hugging Face: [`pjramg/Safe_Unsafe_Test`](https://huggingface.co/datasets/pjramg/Safe_Unsafe_Test). It contains 40 labeled warehouse surveillance videos covering 8 safety classes (5 videos per class).

In [ ]:
import fiftyone as fo
import fiftyone.utils.huggingface as fouh

dataset_name = "pjramg/Safe_Unsafe_Test"

if fo.dataset_exists(dataset_name):
    dataset = fo.load_dataset(dataset_name)
else:
    dataset = fouh.load_from_hub(
        dataset_name,
        persistent=True,
        overwrite=True,
    )

sample = dataset.first()
print(f"Loaded dataset with {len(dataset)} samples. Media type: {sample.media_type}")

## 4. Define Expert Inspector Prompts

The system prompt establishes the domain expertise. The user prompt defines the strict 8-class classification table with structured JSON output.

**Prompt engineering notes** (from the [Cosmos3 Reasoner Prompt Guide](../basic_examples/reasoner_prompt_guide.md)):
- Media (video) always comes **before** text in the user content array
- Chain-of-thought reasoning uses the `<think>` tag format
- Sampling parameters for reasoning: `temperature=0.6, top_p=0.95`

In [ ]:
SYSTEM_INSTRUCTIONS = """You are an expert Industrial Safety Inspector monitoring a manufacturing facility.
Your goal is to classify the video into EXACTLY ONE of the 8 classes defined below.

CRITICAL NEGATIVE CONSTRAINTS (What to IGNORE):
1. IGNORE SITTING WORKERS:
   - If a person is SITTING at a machine board working, this is NOT an intervention class. Ignore them.
   - If a person is SITTING driving a forklift, the driver is NOT the class. Focus only on the LOAD carried.
2. IGNORE BACKGROUND:
   - The facility is old. Do not report hazards based on faded floor markings or unpainted areas.
3. SINGLE OUTPUT:
   - Even if multiple things happen, choose the MOST PROMINENT behavior.
   - Prioritize UNSAFE behaviors over SAFE behaviors if both are present."""

USER_PROMPT_CONTENT = """Analyze the video and output a JSON object. You MUST select the class ID and Label EXACTLY from the table below.

STRICT CLASSIFICATION TABLE (Use these exact IDs and Labels):

| ID | Label | Definition (Ground Truth) | Hazard Status |
| :--- | :--- | :--- | :--- |
| 0 | Safe Walkway Violation | Worker walks OUTSIDE the designated Green Path. | TRUE (Unsafe) |
| 4 | Safe Walkway | Worker walks INSIDE the designated Green Path. | FALSE (Safe) |
| 1 | Unauthorized Intervention | Worker interacts with machine board WITHOUT a green vest. | TRUE (Unsafe) |
| 5 | Authorized Intervention | Worker interacts with machine board WITH a green vest. | FALSE (Safe) |
| 2 | Opened Panel Cover | Machine panel cover is left OPEN after intervention. | TRUE (Unsafe) |
| 6 | Closed Panel Cover | Machine panel cover is CLOSED after intervention. | FALSE (Safe) |
| 3 | Carrying Overload with Forklift | Forklift carries 3 OR MORE blocks. | TRUE (Unsafe) |
| 7 | Safe Carrying | Forklift carries 2 OR FEWER blocks. | FALSE (Safe) |

INSTRUCTIONS:
1. Identify the behavior in the video.
2. Match it to one row in the table above.
3. Output the exact \"ID\" and \"Label\" from that row. Do not invent new labels.

Answer the question using the following format:

<think>
Your reasoning about the observed safety behavior.
</think>

Write your JSON answer immediately after the </think> tag:
{\"prediction_class_id\": [Integer from Table], \"prediction_label\": \"[Exact String from Table]\", \"video_description\": \"[Concise description of the observed action]\", \"hazard_detection\": {\"is_hazardous\": [true/false based on the Hazard Status column], \"temporal_segment\": \"[Start Time - End Time] or null\"}}"""

print(f"System prompt: {len(SYSTEM_INSTRUCTIONS)} chars")
print(f"User prompt:   {len(USER_PROMPT_CONTENT)} chars")

## 5. Inference Function

Uses the OpenAI-compatible API provided by the vLLM server. Video is passed as a `file://` URI (the server has filesystem access via `--allowed-local-media-path`). Frame sampling is configured through `mm_processor_kwargs` following the [Cosmos3 Reasoner Prompt Guide](../basic_examples/reasoner_prompt_guide.md).

In [ ]:
import json
import re
import openai
from pathlib import Path

VLLM_PORT = os.environ.get("VLLM_PORT", "8001")
client = openai.OpenAI(api_key="EMPTY", base_url=f"http://localhost:{VLLM_PORT}/v1")
MODEL = client.models.list().data[0].id
print(f"Connected to vLLM model: {MODEL}")


def run_inference(video_path: str) -> str:
    """Run Cosmos 3 Reasoner inference on a video via the vLLM OpenAI API."""
    video_url = Path(video_path).resolve().as_uri()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
            {
                "role": "user",
                "content": [
                    {"type": "video_url", "video_url": {"url": video_url}},
                    {"type": "text", "text": USER_PROMPT_CONTENT},
                ],
            },
        ],
        max_tokens=4096,
        temperature=0.6,
        top_p=0.95,
        presence_penalty=0.0,
        extra_body={
            "top_k": 20,
            "repetition_penalty": 1.0,
            "mm_processor_kwargs": {"fps": 2, "do_sample_frames": True},
        },
    )
    return response.choices[0].message.content


def parse_json_response(text: str) -> dict:
    """Extract JSON from model output, handling <think> tags and code fences."""
    if "</think>" in text:
        text = text.split("</think>", 1)[1]
    text = re.sub(r"```(?:json)?", "", text).strip().strip("`").strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return json.loads(match.group(0))
    raise json.JSONDecodeError("No JSON object found", text, 0)

## 6. Run Inference on the Dataset

In [ ]:
success_count = 0
error_count = 0

for sample in dataset.iter_samples(progress=True):
    video_path = sample.filepath
    print(f"\nProcessing: {os.path.basename(video_path)}")

    try:
        output_text = run_inference(video_path)
        print(f"Raw output: {output_text[:200]}...")

        json_data = parse_json_response(output_text)

        sample["cosmos3_analysis"] = json_data
        sample["cosmos3_raw_output"] = output_text
        sample["safety_label"] = fo.Classification(
            label=json_data.get("prediction_label", "unknown"),
            confidence=1.0,
        )
        sample.save()
        success_count += 1
        print(f"  -> {json_data.get('prediction_label')} (hazardous={json_data.get('hazard_detection', {}).get('is_hazardous')})")

    except Exception as e:
        print(f"  ERROR: {e}")
        error_count += 1

print(f"\nDone: {success_count} success, {error_count} errors out of {len(dataset)} samples")

## 7. Visualize Results in FiftyOne

In [ ]:
session = fo.launch_app(dataset, port=5151, auto=False)
print(f"FiftyOne App: {session.url}")

## 8. Analyze Results

In [ ]:
hazardous_view = dataset.match(
    {"cosmos3_analysis.hazard_detection.is_hazardous": True}
)
print(f"Hazardous samples: {len(hazardous_view)} / {len(dataset)}")

class_counts = dataset.count_values("safety_label.label")
print("\nPredictions per class:")
for label, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label}: {count}")

## 9. Compare with Ground Truth

In [ ]:
if dataset.has_sample_field("ground_truth"):
    correct = 0
    total = 0
    for sample in dataset.iter_samples():
        if sample.ground_truth and sample.safety_label:
            total += 1
            if sample.ground_truth.label == sample.safety_label.label:
                correct += 1

    if total > 0:
        accuracy = correct / total * 100
        print(f"Accuracy: {accuracy:.1f}% ({correct}/{total})")
else:
    print("No ground_truth field found. Skipping accuracy computation.")

## Results / Expected Output

Below are representative results from a successful run on **Cosmos3-Nano** (single RTX PRO 5000, 24 GB VRAM). All videos are from the open-access [pjramg/Safe_Unsafe_Test](https://huggingface.co/datasets/pjramg/Safe_Unsafe_Test) dataset.

### Evaluation Summary (40 videos)

| Metric | Cosmos3-Nano (1× RTX 5000) |
|--------|----------------------------|
| Exact Class Accuracy | 48.7% (19/39) |
| Category Accuracy | 56.4% (22/39) |
| Hazard Detection | 56.4% (22/39) |
| Avg Inference Time | 38.1s per video |

### Per-Class Accuracy

| Class | Accuracy | Notes |
|-------|----------|-------|
| 3 — Forklift Overload | **100%** (5/5) | Perfect — model counts blocks accurately |
| 7 — Safe Carrying | **100%** (4/4) | Perfect — ≤2 blocks correctly identified |
| 1 — Unauthorized Intervention | **80%** (4/5) | Strong — vest detection reliable |
| 4 — Safe Walkway | **60%** (3/5) | Moderate |
| 5 — Authorized Intervention | **40%** (2/5) | Vest color sometimes missed |
| 2 — Opened Panel Cover | **20%** (1/5) | Hard for Nano |
| 0 — Walkway Violation | **0%** (0/5) | Path boundaries need Super |
| 6 — Closed Panel Cover | **0%** (0/5) | Panel state detection needs Super |

### Successful Predictions with Embedded Videos

#### Class 1 — Unauthorized Intervention (Unsafe) ✅

<video src="https://huggingface.co/datasets/pjramg/Safe_Unsafe_Test/resolve/main/data/1_tr28.mp4" width="640" controls></video>

**Model output:**
```json
{
  "prediction_class_id": 1,
  "prediction_label": "Unauthorized Intervention",
  "video_description": "A worker in a dark uniform approaches a large industrial machine and interacts with its control panel without wearing a green safety vest.",
  "hazard_detection": { "is_hazardous": true, "temporal_segment": "0.0 - 14.0 seconds" }
}
```

> **Chain-of-thought:** *"I see a worker approaching a large industrial machine and interacting with its control panel. The worker is wearing a dark uniform but no green safety vest, which is a clear violation of the required PPE. Because the worker is not authorized by vest..."*

#### Class 2 — Opened Panel Cover (Unsafe) ✅

<video src="https://huggingface.co/datasets/pjramg/Safe_Unsafe_Test/resolve/main/data/2_tr2.mp4" width="640" controls></video>

**Model output:**
```json
{
  "prediction_class_id": 2,
  "prediction_label": "Opened Panel Cover",
  "video_description": "Two workers are seen interacting with a large industrial machine. One worker, wearing a high-visibility vest, is standing and appears to be performing maintenance near the open panel. After their intervention, the machine's panel cover remains open.",
  "hazard_detection": { "is_hazardous": true, "temporal_segment": "0.0 - 12.0 seconds" }
}
```

> **Chain-of-thought:** *"I can see that the machine panel cover is left open after the workers have finished their intervention. This violates safety protocols because an open panel exposes internal components and poses an electrocution or injury risk."*

#### Class 3 — Carrying Overload with Forklift (Unsafe) ✅

<video src="https://huggingface.co/datasets/pjramg/Safe_Unsafe_Test/resolve/main/data/3_tr14.mp4" width="640" controls></video>

**Model output:**
```json
{
  "prediction_class_id": 3,
  "prediction_label": "Carrying Overload with Forklift",
  "video_description": "A forklift is transporting multiple stacked orange containers across the factory floor, moving from left to right while carrying a load that exceeds safe capacity limits.",
  "hazard_detection": { "is_hazardous": true, "temporal_segment": "0.0 - 9.9 seconds" }
}
```

> **Chain-of-thought:** *"The video shows a forklift carrying multiple orange containers. The forklift appears to have three or more stacked containers... The stack is very high and could easily tip over during movement."*

#### Class 4 — Safe Walkway (Safe) ✅

<video src="https://huggingface.co/datasets/pjramg/Safe_Unsafe_Test/resolve/main/data/4_tr39.mp4" width="640" controls></video>

**Model output:**
```json
{
  "prediction_class_id": 4,
  "prediction_label": "Safe Walkway",
  "video_description": "A person is walking through the factory aisle while staying within the designated green safety path. The worker follows the marked walkway correctly.",
  "hazard_detection": { "is_hazardous": false, "temporal_segment": null }
}
```

> **Chain-of-thought:** *"I see a person walking through the factory aisle, and they are clearly staying within the designated green safety path. There are no deviations from the walkway."*

#### Class 5 — Authorized Intervention (Safe) ✅

<video src="https://huggingface.co/datasets/pjramg/Safe_Unsafe_Test/resolve/main/data/5_tr16.mp4" width="640" controls></video>

**Model output:**
```json
{
  "prediction_class_id": 5,
  "prediction_label": "Authorized Intervention",
  "video_description": "A worker in a green safety vest is operating a large industrial machine, pressing buttons on its control panel.",
  "hazard_detection": { "is_hazardous": false, "temporal_segment": null }
}
```

> **Chain-of-thought:** *"A worker in a green safety vest is operating a large industrial machine, pressing buttons on its control panel. The green vest indicates authorized access to the machine controls."*

#### Class 7 — Safe Carrying (Safe) ✅

<video src="https://huggingface.co/datasets/pjramg/Safe_Unsafe_Test/resolve/main/data/7_tr3.mp4" width="640" controls></video>

**Model output:**
```json
{
  "prediction_class_id": 7,
  "prediction_label": "Safe Carrying",
  "video_description": "A forklift is moving a single pallet with two stacked orange pallets across a factory floor. The forklift driver is operating the vehicle within the designated green path.",
  "hazard_detection": { "is_hazardous": false, "temporal_segment": null }
}
```

> **Chain-of-thought:** *"The forklift in the video is moving a single pallet with two stacked orange pallets... The forklift carries 2 or fewer items, which is within the safe limit."*

### Key Findings

1. **Forklift monitoring is production-ready** — 100% accuracy on both overload (class 3) and safe-carrying (class 7), even with Cosmos3-Nano on a consumer GPU.

2. **Intervention detection is strong** — 80% for unauthorized intervention, with the model correctly identifying vest-wearing patterns.

3. **Walkway and panel cover classes need Cosmos3-Super** — distinguishing "inside vs outside" the green path (class 0/4) and detecting open/closed panel states (class 2/6) require the fine-grained spatial understanding of the larger model.

4. **Chain-of-thought reasoning is consistent** — all successful inferences produce structured `<think>` reasoning before committing to the JSON classification, enabling human reviewers to audit the model's decision process.

## Alternative Backends

This recipe uses **vLLM** as the primary backend. The same prompts and conversation format work with:

| Backend | Setup | Best For |
|---------|-------|----------|
| **vLLM** (this notebook) | `vllm serve nvidia/Cosmos3-Super` + `vllm-cosmos3` plugin | Batch processing, custom deployments |
| **NVIDIA NIM** | `docker run nvcr.io/nim/nvidia/cosmos3-reasoner:1.7.0` | Production deployment, no Python setup |
| **Cosmos Framework** | `python -m cosmos_framework.scripts.inference` | Research, single-sample inference |

See the [Reasoner README](../README.md) for setup instructions for each backend.